# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [9]:
import pandas as pd
import numpy as np
import pickle
import time
import os
import optuna
import trackio
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for notebook execution
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error


from sklearn.preprocessing import LabelEncoder

# Add any other imports you need here

# Silence Optuna's per-trial console output
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [10]:
# Load raw datasets
train_df = pd.read_csv('Dataset/train.csv')
test_df  = pd.read_csv('Dataset/test.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
display(train_df.head())
display(test_df.head())

# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
# Ensure you keep all necessary features that will be shown on the UI dashboard.

# Identify target and feature columns
target_col       = 'price'
categorical_cols = train_df.select_dtypes(include='object').columns.tolist()   # pandas 2.x compatible
feature_cols     = [c for c in train_df.columns if c != target_col]

# Fill missing values — median for numeric, 'Unknown' string for categorical
numeric_cols = [c for c in train_df.columns if c not in categorical_cols]
for col in numeric_cols:
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(median_val)

for col in categorical_cols:
    train_df[col] = train_df[col].fillna('Unknown').astype(str)
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna('Unknown').astype(str)

# Encode each categorical column with LabelEncoder fitted on train+test combined
# (fitting on combined values prevents transform errors for unseen test labels)
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    combined_vals = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined_vals)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col]  = le.transform(test_df[col].astype(str))
    encoders[col] = le

# Print learned mappings so the Streamlit app can verify encoding
for col, le in encoders.items():
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"\n{col} mapping: {mapping}")

# TODO: Separate predictors (X) and target (y: 'price')
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_test  = test_df[feature_cols].copy()
y_test  = test_df[target_col].copy() if target_col in test_df.columns else None

print('\nX_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)

# Persist LabelEncoders and feature metadata so Streamlit app uses identical preprocessing
os.makedirs('models', exist_ok=True)
preprocess_artifacts = {
    'encoders':         encoders,
    'categorical_cols': categorical_cols,
    'feature_cols':     feature_cols,
    'target_col':       target_col,
}
with open('models/preprocess_artifacts.pkl', 'wb') as f:
    pickle.dump(preprocess_artifacts, f)
print('Saved -> models/preprocess_artifacts.pkl')

Train shape: (11128, 15)
Test shape : (2782, 15)


,location,city,latitude,longitude,price,numBathrooms,numBalconies,isNegotiable,SecurityDeposit,Status,Size_ft²,BHK,rooms_num,property_type,verification_days
0,Thane West,Mumbai,19.216236,72.980614,27000,2,0,0,0,Unfurnished,530,1,2,Apartment,10.0
1,Thane West,Mumbai,19.271788,72.967789,13500,2,0,0,0,Unfurnished,650,1,1,Apartment,1460.0
2,Airoli,Mumbai,19.143419,72.997307,20000,1,0,0,0,Semi-Furnished,700,1,1,Apartment,730.0
3,Mahipalpur,Delhi,28.547979,77.130135,14000,1,0,0,0,Furnished,1200,0,1,Studio Apartment,150.0
4,Jasola,Delhi,28.529716,77.293137,24500,2,1,0,49000,Unfurnished,1000,1,2,Apartment,180.0


,location,city,latitude,longitude,price,numBathrooms,numBalconies,isNegotiable,SecurityDeposit,Status,Size_ft²,BHK,rooms_num,property_type,verification_days
0,Andheri East,Mumbai,19.122274,72.850876,11000,1,0,0,0,Semi-Furnished,250,0,1,Studio Apartment,1095.000000
1,Thane West,Mumbai,19.211683,73.014488,18000,2,0,0,0,Semi-Furnished,700,1,1,Apartment,30.000000
2,Ghansoli,Mumbai,19.123434,73.003281,45000,2,4,1,150000,Unfurnished,1200,1,2,Apartment,180.000000
3,Andheri East,Mumbai,19.115671,72.882957,45000,2,0,0,0,Furnished,565,1,1,Apartment,17.000000
4,Mira Road East,Mumbai,19.297258,72.865738,26000,2,0,0,0,Furnished,690,1,1,Apartment,0.416667



location mapping: {' Kharadi': np.int64(0), 'AGCR Enclave': np.int64(1), 'Aarya Chanakya Nagar': np.int64(2), 'Abul Fazal Enclave Jamia Nagar': np.int64(3), 'Agalambe': np.int64(4), 'Agripada': np.int64(5), 'Airoli': np.int64(6), 'Ajmeri Gate': np.int64(7), 'Akurdi': np.int64(8), 'Akurli Road': np.int64(9), 'Akurli Road Number 1': np.int64(10), 'Alaknanda': np.int64(11), 'Alandi': np.int64(12), 'Ambegaon Budruk': np.int64(13), 'Ambegaon Pathar': np.int64(14), 'Ambernath East': np.int64(15), 'Ambernath West': np.int64(16), 'Amrita Shergill Marg': np.int64(17), 'Amritpuri': np.int64(18), 'Amrut Nagar': np.int64(19), 'Anand Lok': np.int64(20), 'Anand Nagar': np.int64(21), 'Anand Niketan': np.int64(22), 'Anand Vihar': np.int64(23), 'Andheri East': np.int64(24), 'Andheri West': np.int64(25), 'Ansari Nagar West': np.int64(26), 'Antarli': np.int64(27), 'Ashok Nagar': np.int64(28), 'Ashok Vihar': np.int64(29), 'Aundh': np.int64(30), 'Aundh Gaon': np.int64(31), 'Aurungzeb Road': np.int64(32), 

## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [11]:
# Shared base estimator for all three HPO strategies
rf = RandomForestRegressor(random_state=42)

# TODO: Initialize trackio project/experiment here
# Each HPO method gets its own named trackio run so the dashboard shows
# "Grid Search", "Random Search", "Bayesian Optimization" as separate comparable runs.
TRACKIO_PROJECT = "UrbanNest-RentPrediction"


def _to_python(obj):
    """Recursively convert numpy scalars -> Python natives (required for JSON logging)."""
    if isinstance(obj, dict):
        return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj


def trackio_run(run_name, metrics: dict):
    """Create a dedicated trackio run with run_name and log metrics, then finish it.

    Using separate init/finish per method ensures the dashboard shows
    Grid Search, Random Search and Bayesian Optimization as distinct named runs.
    """
    try:
        trackio.init(
            project=TRACKIO_PROJECT,
            name=run_name,
            resume='allow'          # avoids duplicate-name warnings on rerun
        )
        trackio.log(_to_python(metrics))
        trackio.finish()
        print(f"  [trackio] Logged run: '{run_name}'")
    except Exception as e:
        print(f"  [trackio] '{run_name}' skipped: {e}")


# ── Grid Search Space: EXACTLY 4 x 5 x 3 = 60 combinations ──────────────────
# n_estimators [50,100,150,200]  -> 4 values
# max_depth    [10,15,20,25,30]  -> 5 values
# min_samples_split [2,5,8]      -> 3 values  => 4 * 5 * 3 = 60 total
param_grid_strict = {
    'n_estimators':      [50, 100, 150, 200],
    'max_depth':         [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 8],
}

# Wider continuous distributions for Random Search and Bayesian Optimization
param_dist_wide = {
    'n_estimators':      list(range(50, 201)),
    'max_depth':         list(range(10, 31)),
    'min_samples_split': list(range(2, 11)),
}

cv_folds = 5
scoring  = 'neg_mean_absolute_error'

# Convert to numpy arrays for cross-validator serialization safety
X_train_cv = X_train.to_numpy()
y_train_cv = y_train.to_numpy()

results_summary = {}            # collects best result from each method
grid_history   = []             # (iteration, best_mae_so_far) for plotting
random_history = []
bayes_history  = []

# TODO: 1. Grid Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
print("[1/3] Grid Search: 60 combos x 5 folds = 300 fits ...")
grid_start  = time.time()
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_strict,
    scoring=scoring,
    cv=cv_folds,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_cv, y_train_cv)
grid_time = time.time() - grid_start

# Build the 'best CV MAE so far' convergence curve
grid_iterations = len(param_grid_strict['n_estimators']) * len(param_grid_strict['max_depth']) * len(param_grid_strict['min_samples_split'])  # 60
grid_errors = [-s for s in grid_search.cv_results_['mean_test_score']]
best_so_far = np.inf
for i, err in enumerate(grid_errors, start=1):
    best_so_far = min(best_so_far, err)
    grid_history.append((i, best_so_far))

results_summary['grid'] = {
    'best_params': grid_search.best_params_,
    'best_cv_mae': float(-grid_search.best_score_),
    'time_sec':    float(grid_time),
    'iterations':  grid_iterations,
}
# Log to trackio as a dedicated run named "Grid Search"
trackio_run('Grid Search', {
    'method':       'Grid Search',
    'iterations':   grid_iterations,
    'time_sec':     float(grid_time),
    'best_cv_mae':  float(-grid_search.best_score_),
    'n_estimators': grid_search.best_params_['n_estimators'],
    'max_depth':    grid_search.best_params_['max_depth'],
    'min_samples_split': grid_search.best_params_['min_samples_split'],
})
print(f"  Best Params : {grid_search.best_params_}")
print(f"  Best CV MAE : {-grid_search.best_score_:.2f}  |  Time: {grid_time:.1f}s")


# TODO: 2. Random Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
print("\n[2/3] Random Search: 60 trials x 5 folds ...")
random_start  = time.time()
random_n_iter = 60          # budget matches Grid Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_wide,
    n_iter=random_n_iter,
    scoring=scoring,
    cv=cv_folds,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
random_search.fit(X_train_cv, y_train_cv)
random_time = time.time() - random_start

# Build convergence curve for Random Search
random_errors = [-s for s in random_search.cv_results_['mean_test_score']]
best_so_far = np.inf
for i, err in enumerate(random_errors, start=1):
    best_so_far = min(best_so_far, err)
    random_history.append((i, best_so_far))

results_summary['random'] = {
    'best_params': random_search.best_params_,
    'best_cv_mae': float(-random_search.best_score_),
    'time_sec':    float(random_time),
    'iterations':  random_n_iter,
}
# Log to trackio as a dedicated run named "Random Search"
trackio_run('Random Search', {
    'method':       'Random Search',
    'iterations':   random_n_iter,
    'time_sec':     float(random_time),
    'best_cv_mae':  float(-random_search.best_score_),
    'n_estimators': random_search.best_params_['n_estimators'],
    'max_depth':    random_search.best_params_['max_depth'],
    'min_samples_split': random_search.best_params_['min_samples_split'],
})
print(f"  Best Params : {random_search.best_params_}")
print(f"  Best CV MAE : {-random_search.best_score_:.2f}  |  Time: {random_time:.1f}s")


# TODO: 3. Bayesian Optimization (Optuna) Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
def objective(trial):
    """Optuna objective — returns 5-fold CV MAE (minimised)."""
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 200),
        'max_depth':         trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state': 42,
        'n_jobs': -1,
    }
    model     = RandomForestRegressor(**params)
    cv_scores = cross_val_score(model, X_train_cv, y_train_cv,
                                cv=cv_folds, scoring=scoring, n_jobs=1)
    return float(-cv_scores.mean())     # positive MAE so Optuna minimises it

print("\n[3/3] Bayesian Optimization (Optuna): 60 trials ...")
bayes_start    = time.time()
bayes_n_trials = 60         # budget matches Grid & Random Search
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=bayes_n_trials, show_progress_bar=True)
bayes_time = time.time() - bayes_start

# Build convergence curve from Optuna trial history
best_so_far = np.inf
for i, t in enumerate(study.trials, start=1):
    if t.value is not None:
        best_so_far = min(best_so_far, t.value)
        bayes_history.append((i, best_so_far))

results_summary['bayes'] = {
    'best_params': study.best_params,
    'best_cv_mae': float(study.best_value),
    'time_sec':    float(bayes_time),
    'iterations':  bayes_n_trials,
}
# Log to trackio as a dedicated run named "Bayesian Optimization"
trackio_run('Bayesian Optimization', {
    'method':       'Bayesian Optimization',
    'iterations':   bayes_n_trials,
    'time_sec':     float(bayes_time),
    'best_cv_mae':  float(study.best_value),
    'n_estimators': study.best_params['n_estimators'],
    'max_depth':    study.best_params['max_depth'],
    'min_samples_split': study.best_params['min_samples_split'],
})
print(f"  Best Params : {study.best_params}")
print(f"  Best CV MAE : {study.best_value:.2f}  |  Time: {bayes_time:.1f}s")

print('\n── Summary ──')
display(pd.DataFrame(results_summary).T)

[1/3] Grid Search: 60 combos x 5 folds = 300 fits ...
Fitting 5 folds for each of 60 candidates, totalling 300 fits
* Created new run: Grid Search
* Run finished. Uploading logs to Trackio (please wait...)
  [trackio] Logged run: 'Grid Search'
  Best Params : {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}
  Best CV MAE : 13270.76  |  Time: 333.1s

[2/3] Random Search: 60 trials x 5 folds ...
Fitting 5 folds for each of 60 candidates, totalling 300 fits
* Created new run: Random Search
* Run finished. Uploading logs to Trackio (please wait...)
  [trackio] Logged run: 'Random Search'
  Best Params : {'n_estimators': 142, 'min_samples_split': 2, 'max_depth': 24}
  Best CV MAE : 13299.76  |  Time: 157.0s

[3/3] Bayesian Optimization (Optuna): 60 trials ...


  0%|          | 0/60 [00:00<?, ?it/s]

* Created new run: Bayesian Optimization
* Run finished. Uploading logs to Trackio (please wait...)
  [trackio] Logged run: 'Bayesian Optimization'
  Best Params : {'n_estimators': 186, 'max_depth': 26, 'min_samples_split': 2}
  Best CV MAE : 13271.82  |  Time: 484.2s

── Summary ──


,best_params,best_cv_mae,time_sec,iterations
grid,"{'max_depth': 25, 'min_samples_split': 2, 'n_e...",13270.758152,333.100772,60
random,"{'n_estimators': 142, 'min_samples_split': 2, ...",13299.757428,157.036286,60
bayes,"{'n_estimators': 186, 'max_depth': 26, 'min_sa...",13271.820348,484.234268,60


## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [12]:
# Ensure output directories exist
os.makedirs('plots', exist_ok=True)
os.makedirs('screenshots', exist_ok=True)

# TODO: Generate and save trials_vs_error.png
# X-axis: Number of iterations
# Y-axis: Best CV error found so far
# Overlay Grid, Random, and Bayesian methods on the same plot.
fig, ax = plt.subplots(figsize=(10, 6))

g_x, g_y = zip(*grid_history)
r_x, r_y = zip(*random_history)
b_x, b_y = zip(*bayes_history)

ax.plot(g_x, g_y, label='Grid Search',            color='steelblue',  linewidth=2)
ax.plot(r_x, r_y, label='Random Search',          color='darkorange', linewidth=2)
ax.plot(b_x, b_y, label='Bayesian Opt. (Optuna)', color='seagreen',   linewidth=2)

ax.set_xlabel('Number of Trials / Iterations', fontsize=13)
ax.set_ylabel('Best CV MAE (lower is better)',  fontsize=13)
ax.set_title(
    'Compute Budget vs. Best Cross-Validation Error\n'
    '(Grid Search vs. Random Search vs. Bayesian Optimization)',
    fontsize=14
)
ax.legend(fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('plots/trials_vs_error.png', dpi=150)
plt.close(fig)
print('Saved -> plots/trials_vs_error.png')

# TODO: Generate and save optuna_hyperparameter_space.png
# Scatter showing how Bayesian Optimization explored the hyperparameter space.
n_est  = [t.params['n_estimators']      for t in study.trials if t.value is not None]
m_dep  = [t.params['max_depth']         for t in study.trials if t.value is not None]
mss    = [t.params['min_samples_split'] for t in study.trials if t.value is not None]
values = [t.value                        for t in study.trials if t.value is not None]

fig2, axes = plt.subplots(1, 2, figsize=(14, 5))

sc1 = axes[0].scatter(n_est, m_dep, c=values, cmap='viridis_r', s=70, alpha=0.85)
axes[0].set_xlabel('n_estimators', fontsize=12)
axes[0].set_ylabel('max_depth',    fontsize=12)
axes[0].set_title('n_estimators vs max_depth', fontsize=12)
plt.colorbar(sc1, ax=axes[0], label='CV MAE')

sc2 = axes[1].scatter(n_est, mss, c=values, cmap='viridis_r', s=70, alpha=0.85)
axes[1].set_xlabel('n_estimators',      fontsize=12)
axes[1].set_ylabel('min_samples_split', fontsize=12)
axes[1].set_title('n_estimators vs min_samples_split', fontsize=12)
plt.colorbar(sc2, ax=axes[1], label='CV MAE')

plt.suptitle('Optuna – Hyperparameter Space Exploration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/optuna_hyperparameter_space.png', dpi=150)
plt.close(fig2)
print('Saved -> plots/optuna_hyperparameter_space.png')

Saved -> plots/trials_vs_error.png
Saved -> plots/optuna_hyperparameter_space.png


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [13]:
# TODO: Print the best hyperparameters found by all 3 methods
print("=" * 60)
print("BEST HYPERPARAMETERS FOUND BY EACH METHOD")
print("=" * 60)
for name, res in results_summary.items():
    print(f"{name.upper():20s}: {res['best_params']}")
    print(f"  Best CV MAE : {res['best_cv_mae']:.2f}   |  Time: {res['time_sec']:.1f}s")
print("=" * 60)

# Identify overall winner (lowest 5-fold CV MAE)
best_method = min(results_summary, key=lambda k: results_summary[k]['best_cv_mae'])
best_params = results_summary[best_method]['best_params']
print(f"\n* Winner: {best_method.upper()}")
print(f"  CV MAE  = {results_summary[best_method]['best_cv_mae']:.2f}")
print(f"  Params  = {best_params}")

# TODO: Train the best model found on the full X_train
# Retrain winner on the complete training set (no CV split)
best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_model.fit(X_train.to_numpy(), y_train.to_numpy())
print("\nBest model retrained on full training set.")

# TODO: Evaluate the model on X_test (Report MAE)
y_pred   = best_model.predict(X_test.to_numpy())
test_mae = mean_absolute_error(y_test.to_numpy(), y_pred)
print(f"\nFinal Test MAE (on unseen test.csv): {test_mae:.2f}")

# Log final test result as its own dedicated trackio run
trackio_run('Final-Model', {
    'final_test_mae': float(test_mae),
    'best_method':    best_method,
    'n_estimators':   best_params['n_estimators'],
    'max_depth':      best_params['max_depth'],
    'min_samples_split': best_params['min_samples_split'],
})

# TODO: Save best_model.pkl and any necessary encoders to the models/ folder
# Serialize the trained model to disk for Streamlit inference
os.makedirs('models', exist_ok=True)
model_path = 'models/best_rf_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nSaved -> {model_path}")
print("Saved -> models/preprocess_artifacts.pkl  (from Section 1)")

BEST HYPERPARAMETERS FOUND BY EACH METHOD
GRID                : {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}
  Best CV MAE : 13270.76   |  Time: 333.1s
RANDOM              : {'n_estimators': 142, 'min_samples_split': 2, 'max_depth': 24}
  Best CV MAE : 13299.76   |  Time: 157.0s
BAYES               : {'n_estimators': 186, 'max_depth': 26, 'min_samples_split': 2}
  Best CV MAE : 13271.82   |  Time: 484.2s

* Winner: GRID
  CV MAE  = 13270.76
  Params  = {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}

Best model retrained on full training set.

Final Test MAE (on unseen test.csv): 12422.43
* Created new run: Final-Model
* Run finished. Uploading logs to Trackio (please wait...)
  [trackio] Logged run: 'Final-Model'

Saved -> models/best_rf_model.pkl
Saved -> models/preprocess_artifacts.pkl  (from Section 1)
